# 03 — async/await Desugared: Under the Hood

**Goal:** See definitively that `async def` / `await` is syntactic sugar over generator-based coroutines.

## The timeline

```
Python 2.5 (2006): yield + send()          → generators can receive data
Python 3.3 (2012): yield from              → generator delegation  
Python 3.4 (2014): @asyncio.coroutine      → generators used as coroutines
Python 3.5 (2015): async def / await       → dedicated syntax (same mechanics)
```

## Exercise 3.1 — The three equivalent forms

Write the SAME coroutine in three forms:

**Form 1 — Pure generator (Python 3.3):**
```python
def gen_sleep():
    yield ("SLEEP", 0.1)

def gen_task():
    print("before sleep")
    yield from gen_sleep()
    print("after sleep")
```

**Form 2 — Bridge (Python 3.4):** Use `@types.coroutine` on a generator so it can be used with `await`.

**Form 3 — Modern (Python 3.5+):** `async def` + `await`.

Manually drive ALL three with `.send(None)` and verify they produce the same `("SLEEP", 0.1)` command.

In [ ]:
import types

# Exercise 3.1 — your code here


### Question 3.1
Both forms produce the SAME command and behave identically when driven with `.send()`. What does this tell you about `async`/`await`?

*Your answer:*


## Exercise 3.2 — Inspect the coroutine object

Create both an `async def` coroutine and a generator. Compare them:
- Check `type()`, `hasattr(.send)`, `hasattr(.throw)`, `hasattr(.close)`
- Coroutines have `cr_frame`, generators have `gi_frame` — same concept, different names
- Use `inspect.iscoroutine()` and `inspect.isgenerator()`

Conclusion: same interface, different label.

In [ ]:
import inspect
import asyncio

# Exercise 3.2 — your code here


## Exercise 3.3 — The `__await__` protocol

Any object with `__await__()` can be used with `await`.

Build a `MySleep` class:
- `__init__(self, seconds)` — stores seconds
- `__await__(self)` — a generator that yields `("SLEEP", self.seconds)` and returns a result string

Write an `async def demo()` that does `result = await MySleep(2)` and prints the result.

Drive it manually with `.send(None)` — see the command, then resume.

**Key insight:**
```python
result = await MySleep(2)
# is sugar for:
result = yield from MySleep(2).__await__()
```

In [ ]:
# Exercise 3.3 — your code here


## The Complete Picture

```
                    async def my_func():          
                        result = await other()    
                                                  
                           ↓ desugars to ↓        
                                                  
                    def my_func():                
                        result = yield from other().__await__()
                                                  
                           ↓ which means ↓        
                                                  
    my_func yields to the event loop              
    event loop sees the command (sleep/IO/etc)     
    event loop handles it                         
    event loop .send()s the result back           
    my_func resumes with the result               
```

**You now understand async/await from the ground up.**

Next: Module 02 (Threading) and Module 03 (Multiprocessing) — understand OS-level concurrency so you know WHEN to use each. Then Module 05 builds a FULL event loop with I/O polling.